# Sant Quirze offer candidates: top 10 for a 500k-600k target

This notebook ranks current Sant Quirze listings by combining three signals:

- how well the home fits the 500k-600k target band and the 598k accepted-price anchor,
- how strong the property characteristics are, and
- how likely the listing is to be negotiable based on time online and price history.

Important: there is no direct label for an accepted offer in the database, so the negotiation component is a proxy model learned from observable listing behavior.

In [ ]:
from __future__ import annotations

import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge

DB_PATH = Path("immo_scraper_sant_quirze.db")
TARGET_MIN = 500_000
TARGET_MAX = 600_000
TARGET_MID = (TARGET_MIN + TARGET_MAX) / 2
REFERENCE_ACCEPTED_PRICE = 598_000
NOW_UTC = pd.Timestamp.now(tz="UTC")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

def open_db(path: Path = DB_PATH):
    conn = sqlite3.connect(path)
    conn.row_factory = sqlite3.Row
    return conn

def table_columns(conn, table_name: str) -> list[str]:
    return [row[1] for row in conn.execute(f"PRAGMA table_info({table_name})").fetchall()]

def pick_history_time_column(conn) -> str:
    cols = table_columns(conn, "price_history")
    if "date" in cols:
        return "date"
    if "seen_at" in cols:
        return "seen_at"
    raise ValueError(f"Unsupported price_history schema: {cols}")

def to_utc(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series, utc=True, errors="coerce")

def clamp01(series: pd.Series) -> pd.Series:
    return series.clip(lower=0, upper=1)

def distance_fit(price: pd.Series, center: float, scale: float) -> pd.Series:
    return pd.Series(np.clip(1 - (np.abs(price - center) / scale), 0, 1), index=price.index)

def budget_fit(price: pd.Series) -> pd.Series:
    price = pd.to_numeric(price, errors="coerce")
    band_fit = distance_fit(price, TARGET_MID, 120_000)
    accepted_anchor_fit = distance_fit(price, REFERENCE_ACCEPTED_PRICE, 140_000)
    return 0.6 * band_fit + 0.4 * accepted_anchor_fit

def safe_percentile_rank(series: pd.Series) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce")
    ranked = values.rank(pct=True, method="average")
    if ranked.notna().any():
        return ranked.fillna(ranked.median())
    return pd.Series(0.5, index=series.index)

def safe_median(series: pd.Series, default: float = 0.0) -> float:
    values = pd.to_numeric(series, errors="coerce")
    finite_values = values[np.isfinite(values)]
    if len(finite_values) == 0:
        return default
    return float(np.median(finite_values))

In [ ]:
with open_db() as conn:
    history_time_col = pick_history_time_column(conn)
    properties = pd.read_sql_query("SELECT * FROM properties", conn)
    excluded_title_mask = properties["title"].fillna("").str.contains(
        r"Centre|Estació|Monestir|Nau|El\s+Coll|Sant Francesc|Terreny\b",
        case=False,
        na=False,
        regex=True,
    )
    properties = properties.loc[~excluded_title_mask].reset_index(drop=True)
    history = pd.read_sql_query(
        f"SELECT property_id, price, {history_time_col} AS history_ts FROM price_history",
        conn,
    )

properties["first_seen"] = to_utc(properties["first_seen"])
properties["last_seen"] = to_utc(properties["last_seen"])
for col in ["price", "price_first_seen", "rooms", "bathrooms", "sqm", "year_built", "has_pool", "has_ac", "terrace", "elevator", "parking", "similarity_score", "latitude", "longitude"]:
    if col in properties.columns:
        properties[col] = pd.to_numeric(properties[col], errors="coerce")

history["history_ts"] = to_utc(history["history_ts"])
history["price"] = pd.to_numeric(history["price"], errors="coerce")
history = history.dropna(subset=["property_id", "price", "history_ts"]).sort_values(["property_id", "history_ts", "price"]).reset_index(drop=True)

def summarize_history(group: pd.DataFrame) -> pd.Series:
    group = group.sort_values("history_ts").reset_index(drop=True)
    diffs = group["price"].diff()
    last_change_ts = group.loc[diffs.fillna(0).ne(0), "history_ts"].max()
    return pd.Series({
        "history_rows": len(group),
        "history_first_price": group["price"].iloc[0],
        "history_last_price": group["price"].iloc[-1],
        "history_min_price": group["price"].min(),
        "history_max_price": group["price"].max(),
        "price_change_events": int(diffs.fillna(0).ne(0).sum()),
        "price_drop_events": int((diffs < 0).sum()),
        "price_increase_events": int((diffs > 0).sum()),
        "history_span_days": (group["history_ts"].iloc[-1] - group["history_ts"].iloc[0]).days if len(group) > 1 else 0,
        "days_since_last_price_change": (NOW_UTC - last_change_ts).days if pd.notna(last_change_ts) else np.nan,
    })

history_features = history.groupby("property_id", group_keys=False).apply(summarize_history, include_groups=False).reset_index()
features = properties.merge(history_features, on="property_id", how="left")

features["days_online"] = (NOW_UTC - features["first_seen"]).dt.total_seconds() / 86400
features["days_since_last_seen"] = (NOW_UTC - features["last_seen"]).dt.total_seconds() / 86400
features["price_per_sqm"] = features["price"] / features["sqm"]
features["discount_from_first_seen_pct"] = np.where(
    features["price_first_seen"].gt(0),
    (features["price_first_seen"] - features["price"]) / features["price_first_seen"],
    np.nan,
)
features["discount_from_history_peak_pct"] = np.where(
    features["history_max_price"].gt(0),
    (features["history_max_price"] - features["price"]) / features["history_max_price"],
    np.nan,
)
features["price_relative_to_accepted_mean_pct"] = np.where(
    REFERENCE_ACCEPTED_PRICE > 0,
    1 - (np.abs(features["price"] - REFERENCE_ACCEPTED_PRICE) / REFERENCE_ACCEPTED_PRICE),
    np.nan,
)

for col in ["history_rows", "price_change_events", "price_drop_events", "price_increase_events", "history_span_days", "days_since_last_price_change"]:
    features[col] = features[col].fillna(0)

features["budget_fit_score"] = budget_fit(features["price"])
features["quality_score"] = (
    0.22 * safe_percentile_rank(features["sqm"]) +
    0.16 * safe_percentile_rank(features["rooms"]) +
    0.12 * safe_percentile_rank(features["bathrooms"]) +
    0.10 * safe_percentile_rank(features["year_built"]) +
    0.10 * safe_percentile_rank(features["similarity_score"]) +
    0.08 * features["has_pool"].fillna(0) +
    0.08 * features["has_ac"].fillna(0) +
    0.07 * features["terrace"].fillna(0) +
    0.05 * features["elevator"].fillna(0) +
    0.04 * features["parking"].fillna(0) +
    0.08 * (1 - safe_percentile_rank(features["price_per_sqm"]))
).clip(0, 1)

features["negotiation_proxy"] = (
    0.35 * safe_percentile_rank(features["days_online"]) +
    0.30 * safe_percentile_rank(features["discount_from_first_seen_pct"].clip(lower=0)) +
    0.15 * safe_percentile_rank(features["price_change_events"]) +
    0.10 * safe_percentile_rank(features["days_since_last_price_change"]) +
    0.10 * safe_percentile_rank(features["history_span_days"])
).clip(0, 1)

sant_quirze_mask = features["city"].fillna("").str.contains("Sant Quirze", case=False, na=False)
active_mask = features["status"].fillna("").str.lower().eq("active")
candidate_mask = sant_quirze_mask & active_mask & features["price"].between(450_000, 700_000, inclusive="both")

print(f"Properties loaded: {len(properties):,}")
print(f"Removed by title filter: {int(excluded_title_mask.sum()):,}")
print(f"History rows loaded: {len(history):,}")
print(f"Sant Quirze active candidates in the 450k-700k soft band: {int(candidate_mask.sum()):,}")
features[["property_id", "source", "title", "price", "sqm", "rooms", "bathrooms", "days_online", "price_change_events", "budget_fit_score", "quality_score", "negotiation_proxy"]].head(5)

In [ ]:
numeric_features = [
    "price",
    "price_first_seen",
    "sqm",
    "rooms",
    "bathrooms",
    "year_built",
    "latitude",
    "longitude",
    "has_pool",
    "has_ac",
    "terrace",
    "elevator",
    "parking",
    "similarity_score",
    "history_rows",
    "price_change_events",
    "price_drop_events",
    "price_increase_events",
    "history_span_days",
    "days_online",
    "days_since_last_seen",
    "days_since_last_price_change",
    "price_per_sqm",
    "discount_from_first_seen_pct",
    "discount_from_history_peak_pct",
    "price_relative_to_accepted_mean_pct",
    "budget_fit_score",
    "quality_score",
]
categorical_features = ["source", "property_type", "operation", "city", "district", "neighborhood", "energy_rating", "floor", "orientation"]

model_frame = features.copy()
target = model_frame["negotiation_proxy"].fillna(model_frame["negotiation_proxy"].median())

feature_frame = model_frame[numeric_features + categorical_features].copy()
for col in numeric_features:
    feature_frame[col] = pd.to_numeric(feature_frame[col], errors="coerce")
    feature_frame[col] = feature_frame[col].fillna(safe_median(feature_frame[col]))

for col in categorical_features:
    feature_frame[col] = feature_frame[col].astype("string").fillna("missing")

for col in numeric_features:
    mean = feature_frame[col].mean()
    std = feature_frame[col].std(ddof=0)
    if not np.isfinite(std) or std == 0:
        std = 1.0
    feature_frame[col] = (feature_frame[col] - mean) / std

design_matrix = pd.get_dummies(feature_frame, columns=categorical_features, dummy_na=False)
design_matrix = design_matrix.astype(float)
design_matrix.insert(0, "intercept", 1.0)

X = design_matrix.to_numpy()
y = pd.to_numeric(target, errors="coerce").fillna(target.median()).to_numpy(dtype=float)

ridge_alpha = 5.0
penalty = np.eye(X.shape[1])
penalty[0, 0] = 0.0
sk_model = Ridge(alpha=ridge_alpha, fit_intercept=False)
sk_model.fit(X, y)
raw_scores = sk_model.predict(X)
score_min = np.nanmin(raw_scores) if np.isfinite(raw_scores).any() else np.nan
score_max = np.nanmax(raw_scores) if np.isfinite(raw_scores).any() else np.nan
if np.isfinite(score_min) and np.isfinite(score_max) and score_max > score_min:
    negotiation_model_score = (raw_scores - score_min) / (score_max - score_min)
else:
    negotiation_model_score = np.clip(np.nan_to_num(raw_scores, nan=0.5), 0, 1)

features["negotiation_model_score"] = np.clip(negotiation_model_score, 0, 1)
features["final_score"] = (
    0.45 * features["quality_score"] +
    0.35 * features["budget_fit_score"] +
    0.20 * features["negotiation_model_score"]
).clip(0, 1)

recommendations = features.loc[candidate_mask].copy()
if len(recommendations) < 10:
    fallback_mask = sant_quirze_mask & active_mask & features["price"].between(400_000, 750_000, inclusive="both")
    recommendations = features.loc[fallback_mask].copy()

def explain_row(row: pd.Series) -> str:
    reasons = []
    if row["price"] <= TARGET_MAX:
        reasons.append("within target band")
    else:
        reasons.append("near target band")
    if row["days_online"] >= 60:
        reasons.append("long time online")
    if row["price_drop_events"] > 0:
        reasons.append(f"{int(row['price_drop_events'])} price cut(s)")
    if pd.notna(row["sqm"]) and row["sqm"] >= 100:
        reasons.append("strong size")
    if pd.notna(row["similarity_score"]) and row["similarity_score"] >= features["similarity_score"].median():
        reasons.append("good profile match")
    return ", ".join(reasons[:3])

top100 = (
    recommendations
    .sort_values("final_score", ascending=False)
    .head(100)
    .assign(why_it_ranks=lambda df: df.apply(explain_row, axis=1))
)

display_columns = [
    "property_id",
    "source",
    "title",
    "price",
    "sqm",
    "rooms",
    "bathrooms",
    "days_online",
    "history_rows",
    "price_drop_events",
    "budget_fit_score",
    "quality_score",
    "negotiation_model_score",
    "final_score",
    "why_it_ranks",
    "url",
]

top100_display = top100[display_columns].copy()
top100_display["price"] = top100_display["price"].round(0).astype("Int64")
for col in ["sqm", "rooms", "bathrooms", "days_online", "history_rows", "price_drop_events"]:
    top100_display[col] = top100_display[col].round(2)
for col in ["budget_fit_score", "quality_score", "negotiation_model_score", "final_score"]:
    top100_display[col] = top100_display[col].round(3)

print(f"Final recommendation pool: {len(recommendations):,} listings")
print("Top 100 ranked candidates:")
top100_display

## How to read the ranking

The final score blends quality, budget fit, and a proxy negotiability score learned from listing history. That means the top 10 is not a promise of an accepted offer; it is a shortlist of listings that look strongest for a 500k-600k negotiation strategy in Sant Quirze.

If you want to tune the list, the easiest knobs are the target band, the accepted-price anchor, and the weights in `final_score`.

In [ ]:
top100_display.to_excel("sant_quirze_offer_candidates_500_600k.xlsx", index=False)